In [0]:
%run ../00-common/config

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import time, uuid

# parametri i cutoff, ndan history nga increment
dbutils.widgets.text("p_cutoff_date", "2017-12-31")
cutoff = dbutils.widgets.get("p_cutoff_date")

# parametri i modalitetit: 'history' (<=cutoff) ose 'increment' (>cutoff)
dbutils.widgets.dropdown("p_mode", "history", ["history", "increment"])
mode = dbutils.widgets.get("p_mode")

run_id = str(uuid.uuid4())[:8]
print(f"Run {run_id}: mode={mode}, cutoff={cutoff}")

In [0]:
from pyspark.sql.window import Window

start = time.time()

# lexo orders silver, filtro sipas modalitetit
orders = spark.table(f"{catalog_name}.{silver_schema}.orders")

if mode == "history":
    orders_slice = orders.filter(F.col("order_purchase_timestamp") <= F.lit(cutoff))
else:  # increment
    orders_slice = orders.filter(F.col("order_purchase_timestamp") > F.lit(cutoff))

rows_read = orders_slice.count()
print(f"Orders in this slice: {rows_read:,}")

# customers per customer_unique_id (si te gold)
customers = spark.table(f"{catalog_name}.{silver_schema}.customers").select(
    "customer_id", "customer_unique_id", "customer_zip_code_prefix")

# ndertimi i fact_orders per kete slice (e njejta logjike si 06_fact_orders, por filtruar)
reviews = spark.table(f"{catalog_name}.{silver_schema}.order_reviews")
w_review = Window.partitionBy("order_id").orderBy(
    F.desc_nulls_last("review_answer_timestamp"),
    F.desc_nulls_last("review_creation_date"))
review_per_order = (reviews
    .withColumn("rn", F.row_number().over(w_review))
    .filter(F.col("rn") == 1)
    .select("order_id", "review_score"))

fact_slice = (orders_slice
              .join(customers, on="customer_id", how="left")
    .join(review_per_order, on="order_id", how="left")
    .withColumn("date_key", F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int"))
    .withColumn("is_on_time",
        F.when(F.col("order_delivered_customer_date").isNull(), None)
         .otherwise(F.col("order_delivered_customer_date") <= F.col("order_estimated_delivery_date")))
    .withColumn("is_delivered", F.col("order_status") == "delivered")
    .select("order_id", "customer_id", "customer_unique_id", "customer_zip_code_prefix", "date_key", "order_status",
            "review_score", "is_delivered", "is_on_time",
            "approval_hours", "handling_hours", "shipping_days",
            "total_delivery_days", "delivery_variance_days"))

In [0]:
target = f"{catalog_name}.{gold_schema}.fact_orders"

# MERGE: idempotent, riekzekutimi s'dyfishon (perditeson ekzistuesit, shton te rinjte)
if spark.catalog.tableExists(target):
    dt = DeltaTable.forName(spark, target)
    (dt.alias("t")
        .merge(fact_slice.alias("s"), "t.order_id = s.order_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    rows_written = fact_slice.count()
else:
    fact_slice.write.format("delta").saveAsTable(target)
    rows_written = fact_slice.count()

print(f"Merged {rows_written:,} rows into fact_orders")

In [0]:
# perditeso watermark-un
max_date = orders_slice.agg(F.max("order_purchase_timestamp")).collect()[0][0]
if max_date:
    wm = spark.createDataFrame(
        [("olist_pipeline", max_date.date())],
        ["pipeline_name", "last_processed_date"]
    ).withColumn("updated_at", F.current_timestamp())

    wm_target = f"{catalog_name}.{control_schema}.watermark"
    dt_wm = DeltaTable.forName(spark, wm_target)
    (dt_wm.alias("t").merge(wm.alias("s"), "t.pipeline_name = s.pipeline_name")
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

# log run-in
duration = time.time() - start
log_row = spark.createDataFrame(
    [(run_id, "gold", "fact_orders", rows_read, rows_written, 0, float(duration))],
    ["run_id", "layer", "table_name", "rows_read", "rows_written", "rows_quarantined", "duration_seconds"]
).withColumn("run_timestamp", F.current_timestamp())

log_row.write.format("delta").mode("append").saveAsTable(f"{catalog_name}.{control_schema}.run_log")
print(f"Logged run {run_id}: read={rows_read}, written={rows_written}, {duration:.1f}s")

In [0]:
print("=== WATERMARK ===")
spark.table(f"{catalog_name}.{control_schema}.watermark").show(truncate=False)

print("=== RUN LOG ===")
spark.table(f"{catalog_name}.{control_schema}.run_log").show(truncate=False)